In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader

sentences = ["what is statquest <EOS> awesome",
             "statquest is what <EOS> awesome"]

token_to_id = {'what': 0,
               'is': 1,
               'statquest': 2,
               'awesome': 3,
               '<EOS>': 4} # <EOS> = end of sequence

id_to_token = dict(map(reversed, token_to_id.items()))

inputs = torch.tensor([[token_to_id[token] for token in sentence.split()]
                       for sentence in sentences])
# what is statquest <EOS> awesome
# statquest is what <EOS> awesome

labels = torch.tensor([[token_to_id[token] for token in sentence.split()[1:]] + [token_to_id['<EOS>']]
                       for sentence in sentences])
# is statquest <EOS> awesome <EOS>
# is what <EOS> awesome <EOS>

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

In [12]:
from importlib import reload
from src import transformer
reload(transformer)
from src.transformer import DecoderOnlyTransformer

import lightning as L

max_length = 6
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=2, max_len=max_length)

In [15]:
### Check

eos_id = token_to_id['<EOS>']

model_input = ["what is statquest <EOS>",
               "statquest is what <EOS>"]

model_input = torch.tensor([[token_to_id[token] for token in sentence.split()]
                           for sentence in model_input])
input_length = model_input.size(dim=1)
predicted_ids = torch.tensor([])
for _ in range(input_length, max_length):
  predictions = model(model_input)
  predicted_id = torch.argmax(predictions[:, -1:], dim=-1)
  # predicted_id: [batch_size, 1]
  predicted_ids = torch.cat([predicted_ids, predicted_id], dim=-1)

  model_input = torch.cat([model_input, predicted_id], dim=-1)

print("Predicted Tokens:")
for batch in predicted_ids:
  print(" ".join([id_to_token[id.item()] for id in batch]))

Predicted Tokens:
statquest is
statquest is


In [16]:
### Training

trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 10     | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 12     | train
3 | fc_layer       | Linear           | 15     | train
4 | loss           | CrossEntropyLoss | 0      | train
------------------------------------------------------------
37        Trainable params
0         Non-trainable params
37        Total params
0.000     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.
